# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. The dataset source is provided via a Croissant schema URL, and all data entities are referenced by their unique `@id` fields following Croissant and FAIR practices.


### Dataset Source
Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

List all record sets, along with their fields, using their `@id` identifiers. This helps identify what structured data is available and precisely reference fields in data extraction.

> **Note:** All references to record sets and fields are by their `@id`s as defined by Croissant.

In [ ]:
# Explore the available record sets
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print(f"Found {len(record_sets)} record sets:\n")
    for rs in record_sets:
        print(f"Record set @id: {rs.id} | name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field @id: {field.id} | name: {field.name}")
        print()

## 3. Data Extraction

Load tabular records from specific record sets into Pandas DataFrames, referencing by `@id`.

If you have multiple record sets, you may extract each; here, we will load all available record sets into separate DataFrames.

In [ ]:
# Ids of all available record sets
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Shape: {dataframes[rs_id].shape}")
    if len(dataframes[rs_id].columns) > 0:
        print("Columns:", dataframes[rs_id].columns.tolist())
    else:
        print("No columns found.")
    print()

# For demonstration, print sample from the first record set
if dataframes:
    demo_rs_id = record_set_ids[0] if record_set_ids else None
    print(f"Example records from {demo_rs_id}:")
    display(dataframes[demo_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Process one record set: filter, normalize, and group fields using their `@id`s.

> *Note*: All field/column selections and groupings below use their `@id`. Replace the placeholder IDs with those specific to your dataset after cell 2 if required.

In [ ]:
# EDA using one available record set (example: first record set)
if dataframes:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    
    # List available numeric fields by inspecting dtypes
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields available in {rs_id}: {numeric_candidates}")
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        # Example threshold: use median + 1 as a demo threshold
        threshold = float(df[numeric_field_id].median()) + 1 if not df[numeric_field_id].empty else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold}: {filtered_df.shape[0]} found.")
        display(filtered_df.head())
        # Normalizing field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' values:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Attempt grouping by another field (preferably a categorical one)
        candidate_group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = candidate_group_fields[0] if candidate_group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped statistics by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print(f"No numeric fields available in {rs_id} for numeric EDA.")

## 5. Visualization

Visualize data distributions or relationships within the selected record set. This section uses fields referenced by `@id` and pandas plotting. Modify as needed for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    numeric_field_id = numeric_candidates[0]
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in records set {rs_id}")
    plt.show()
    # If a group field exists, plot grouped boxplot
    if candidate_group_fields:
        group_field_id = candidate_group_fields[0]
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id} in {rs_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've loaded and explored the FAIR² dataset using `mlcroissant`, focusing on using entity `@id` fields for robust, schema-driven data handling. After extracting records, we demonstrated basic EDA and visualization. You can extend this workflow to deeper statistical analysis, modeling, or further integration of Croissant's metadata-aware capabilities — always referencing data entities by their `@id`s to maintain reproducibility and clarity.